<a href="https://colab.research.google.com/github/Lyv-ux/DI_Bootcamp/blob/main/W8D2_DailyChallenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ============================
# BARISTABOT - COMPLETE VERSION
# ============================

!pip install -U langchain-google-genai

from typing import Annotated, Literal
from typing_extensions import TypedDict
from collections.abc import Iterable
from random import randint
from pprint import pprint

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages.ai import AIMessage
from langchain_core.messages.tool import ToolMessage


# ============================
# STATE
# ============================

class OrderState(TypedDict):
    messages: Annotated[list, add_messages]
    order: list[str]
    finished: bool


# ============================
# SYSTEM INSTRUCTIONS
# ============================

BARISTABOT_SYSINT = (
    "system",
    "You are a BaristaBot, an interactive cafe ordering system. "
    "Only discuss menu items. Use tools when appropriate. "
    "Confirm orders before placing them."
)

WELCOME_MSG = "Welcome to the BaristaBot cafe. Type `q` to quit. How may I serve you today?"


# ============================
# GEMINI MODEL
# ============================

from google.colab import userdata

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-latest", google_api_key=userdata.get('GOOGLE_API_KEY'))


# ============================
# MENU TOOL
# ============================

@tool
def get_menu() -> str:
    """Provide the latest up-to-date menu."""
    return """
    MENU:
    Coffee Drinks:
    Espresso
    Americano
    Cold Brew

    Coffee Drinks with Milk:
    Latte
    Cappuccino
    Cortado
    Macchiato
    Mocha
    Flat White

    Tea Drinks:
    English Breakfast Tea
    Green Tea
    Earl Grey

    Tea Drinks with Milk:
    Chai Latte
    Matcha Latte
    London Fog

    Other Drinks:
    Steamer
    Hot Chocolate
    """


# ============================
# ORDER TOOLS
# ============================

@tool
def add_to_order(drink: str, modifiers: Iterable[str]) -> str:
    """Adds the specified drink to the customer's order."""
    pass


@tool
def confirm_order() -> str:
    """Asks the customer if the order is correct."""
    pass


@tool
def get_order() -> str:
    """Returns the user's order so far."""
    pass


@tool
def clear_order():
    """Removes all items from the user's order."""
    pass


@tool
def place_order() -> int:
    """Sends the order to the barista for fulfillment."""
    pass


# ============================
# HUMAN NODE
# ============================

def human_node(state: OrderState) -> OrderState:
    last_msg = state["messages"][-1]
    print("Model:", last_msg.content)

    user_input = input("User: ")

    if user_input in {"q", "quit", "exit", "goodbye"}:
        state["finished"] = True

    return state | {"messages": [("user", user_input)]}


# ============================
# CHATBOT NODE
# ============================

def chatbot_with_tools(state: OrderState) -> OrderState:

    defaults = {"order": [], "finished": False}

    if state["messages"]:
        new_output = llm_with_tools.invoke(
            [BARISTABOT_SYSINT] + state["messages"]
        )
    else:
        new_output = AIMessage(content=WELCOME_MSG)

    return defaults | state | {"messages": [new_output]}


# ============================
# ROUTING FUNCTIONS
# ============================

def maybe_exit_human_node(
    state: OrderState
) -> Literal["chatbot", "__end__"]:

    if state.get("finished", False):
        return END
    else:
        return "chatbot"


def maybe_route_to_tools(state: OrderState) -> str:

    if not (msgs := state.get("messages", [])):
        raise ValueError(f"No messages found when parsing state: {state}")

    msg = msgs[-1]

    if state.get("finished", False):
        return END

    elif hasattr(msg, "tool_calls") and len(msg.tool_calls) > 0:

        if any(
            tool["name"] in tool_node.tools_by_name.keys()
            for tool in msg.tool_calls
        ):
            return "tools"
        else:
            return "ordering"

    else:
        return "human"


# ============================
# ORDER NODE
# ============================

def order_node(state: OrderState) -> OrderState:

    tool_msg = state["messages"][-1]

    order = state.get("order", []).copy()

    outbound_msgs = []

    order_placed = False

    for tool_call in tool_msg.tool_calls:

        if tool_call["name"] == "add_to_order":

            modifiers = tool_call["args"].get("modifiers", [])

            modifier_str = (
                ", ".join(modifiers)
                if modifiers
                else "no modifiers"
            )

            order.append(
                f'{tool_call["args"]["drink"]} ({modifier_str})'
            )

            response = "\n".join(order)

        elif tool_call["name"] == "confirm_order":

            print("Your order:")

            if not order:
                print("  (no items)")

            for drink in order:
                print(f"  {drink}")

            response = input("Is this correct? ")

        elif tool_call["name"] == "get_order":

            response = "\n".join(order) if order else "(no order)"

        elif tool_call["name"] == "clear_order":

            order.clear()
            response = None

        elif tool_call["name"] == "place_order":

            order_text = "\n".join(order)

            print("Sending order to kitchen!")
            print(order_text)

            order_placed = True

            response = randint(1, 5)

        else:
            raise NotImplementedError(
                f'Unknown tool call: {tool_call["name"]}'
            )

        outbound_msgs.append(
            ToolMessage(
                content=str(response),
                name=tool_call["name"],
                tool_call_id=tool_call["id"],
            )
        )

    return {
        "messages": outbound_msgs,
        "order": order,
        "finished": order_placed,
    }


# ============================
# TOOLS
# ============================

auto_tools = [get_menu]

tool_node = ToolNode(auto_tools)

order_tools = [
    add_to_order,
    confirm_order,
    get_order,
    clear_order,
    place_order,
]

llm_with_tools = llm.bind_tools(
    auto_tools + order_tools
)


# ============================
# GRAPH
# ============================

graph_builder = StateGraph(OrderState)

graph_builder.add_node("chatbot", chatbot_with_tools)
graph_builder.add_node("human", human_node)
graph_builder.add_node("tools", tool_node)
graph_builder.add_node("ordering", order_node)

graph_builder.add_conditional_edges(
    "chatbot",
    maybe_route_to_tools
)

graph_builder.add_conditional_edges(
    "human",
    maybe_exit_human_node
)

graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge("ordering", "chatbot")

graph_builder.add_edge(START, "chatbot")

graph_with_order_tools = graph_builder.compile()


# ============================
# RUN
# ============================

config = {"recursion_limit": 100}

state = graph_with_order_tools.invoke(
    {"messages": []},
    config
)

pprint(state)

SecretNotFoundError: Secret GOOGLE_API_KEY does not exist.